# Task 2 - Decision Trees

Dans ce notebook, nous allons implémenter un modèle d'arbres de décision, un algorithme intuitif et puissant en machine learning supervisé pour les problèmes de classification et de régression.

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn import tree
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
%matplotlib inline
plt.style.use('seaborn')

## 1. Chargement et exploration des données

Pour cette démonstration, nous utiliserons le jeu de données du cancer du sein de Wisconsin, qui est un problème de classification binaire classique.

In [ ]:
from sklearn.datasets import load_breast_cancer

# Chargement des données du cancer du sein de Wisconsin
breast_cancer = load_breast_cancer()

# Création d'un DataFrame pandas
df = pd.DataFrame(data=breast_cancer.data, columns=breast_cancer.feature_names)
df['target'] = breast_cancer.target

df.head()

In [ ]:
# Exploration des données
print("Shape of dataset:", df.shape)
print("\nInfo of dataset:")
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

In [ ]:
# Vérification de la distribution des classes (variable cible)
target_column = df.columns[-1]
print(f"Variable cible: {target_column}")
print("\nDistribution des classes:")
print(df[target_column].value_counts())

# Visualisation de la distribution des classes
plt.figure(figsize=(8, 6))
df[target_column].value_counts().plot(kind='bar')
plt.title('Distribution des classes')
plt.xlabel('Classes')
plt.ylabel('Nombre d\'échantillons')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

## 2. Prétraitement des données

Séparation des caractéristiques (features) et de la variable cible.

In [ ]:
# Séparation des caractéristiques (features) et de la variable cible
X = df.iloc[:, :-1]  # Toutes les colonnes sauf la dernière
y = df.iloc[:, -1]   # Dernière colonne (variable cible)

print(f"Dimensions des caractéristiques: {X.shape}")
print(f"Dimensions de la variable cible: {y.shape}")

## 3. Division des données

Division des données en ensembles d'entraînement et de test.

In [ ]:
# Division des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape}")
print(f"Taille de l'ensemble de test: {X_test.shape}")

## 4. Entraînement du modèle

Création et entraînement du modèle d'arbre de décision.

In [ ]:
# Création et entraînement de l'arbre de décision
# Paramètres initiaux
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

## 5. Prédiction et évaluation du modèle

Prédiction sur l'ensemble de test et évaluation des performances du modèle.

In [ ]:
# Prédiction sur l'ensemble de test
y_pred = dt_classifier.predict(X_test)

# Affichage des premières prédictions
results_df = pd.DataFrame({'Valeurs réelles': y_test, 'Prédictions': y_pred})
print("Comparaison des valeurs réelles et prédites:")
print(results_df.head(10))

In [ ]:
# Évaluation du modèle
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du modèle: {accuracy:.4f}")

# Rapport de classification détaillé
print("\nRapport de classification:")
print(classification_report(y_test, y_pred))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion')
plt.xlabel('Prédictions')
plt.ylabel('Valeurs réelles')
plt.show()

## 6. Visualisation de l'arbre de décision

Visualisation graphique de l'arbre résultant pour interpréter les règles de décision.

In [ ]:
# Visualisation de l'arbre de décision (simplifiée)
plt.figure(figsize=(20, 10))
plot_tree(dt_classifier, 
          feature_names=X.columns, 
          class_names=['Maligne', 'Bénigne'], 
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title('Arbre de décision')
plt.show()

## 7. Évaluation de la complexité de l'arbre

Analyse de la profondeur et du nombre de feuilles pour comprendre la complexité du modèle.

In [ ]:
# Informations sur l'arbre
print(f"Profondeur de l'arbre: {dt_classifier.get_depth()}")
print(f"Nombre de feuilles: {dt_classifier.get_n_leaves()}")
print(f"Nombre de noeuds: {dt_classifier.tree_.node_count}")

## 8. Pruning (élagage) pour éviter le sur-apprentissage

Application de techniques de pruning pour améliorer la généralisation du modèle.

In [ ]:
# Création d'un arbre avec contraintes pour éviter le sur-apprentissage
dt_pruned = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,           # Limite la profondeur maximale
    min_samples_split=20,  # Nombre minimum d'échantillons pour diviser un noeud
    min_samples_leaf=10,   # Nombre minimum d'échantillons dans une feuille
    max_features='sqrt'    # Nombre maximum de caractéristiques à considérer pour chaque division
)

# Entraînement du modèle élagué
dt_pruned.fit(X_train, y_train)

# Prédiction avec le modèle élagué
y_pred_pruned = dt_pruned.predict(X_test)

# Évaluation du modèle élagué
accuracy_pruned = accuracy_score(y_test, y_pred_pruned)
print(f"Précision du modèle élagué: {accuracy_pruned:.4f}")
print(f"Précision du modèle original: {accuracy:.4f}")

print("\nRapport de classification du modèle élagué:")
print(classification_report(y_test, y_pred_pruned))

In [ ]:
# Comparaison des matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Matrice de confusion du modèle original
cm_original = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_original, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matrice de confusion - Modèle original')
axes[0].set_xlabel('Prédictions')
axes[0].set_ylabel('Valeurs réelles')

# Matrice de confusion du modèle élagué
cm_pruned = confusion_matrix(y_test, y_pred_pruned)
sns.heatmap(cm_pruned, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Matrice de confusion - Modèle élagué')
axes[1].set_xlabel('Prédictions')
axes[1].set_ylabel('Valeurs réelles')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation de l'arbre élagué
plt.figure(figsize=(20, 10))
plot_tree(dt_pruned, 
          feature_names=X.columns, 
          class_names=['Maligne', 'Bénigne'], 
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title('Arbre de décision élagué')
plt.show()

## 9. Importance des caractéristiques

Analyse de l'importance des différentes caractéristiques dans la prise de décision.

In [ ]:
# Importance des caractéristiques
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_classifier.feature_importances_
}).sort_values('Importance', ascending=False)

# Visualisation des 10 caractéristiques les plus importantes
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(10)
plt.barh(range(len(top_features)), top_features['Importance'])
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance')
plt.title('Importance des caractéristiques (Top 10)')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.show()

print("Caractéristiques les plus importantes:")
print(feature_importance.head(10))

## Conclusion

Dans ce notebook, nous avons implémenté un modèle d'arbres de décision :
- Chargement et exploration des données du cancer du sein
- Prétraitement des données
- Division des données en ensembles d'entraînement et de test
- Entraînement du modèle d'arbre de décision
- Évaluation du modèle avec plusieurs métriques
- Visualisation de l'arbre résultant
- Application de techniques de pruning pour éviter le sur-apprentissage
- Analyse de l'importance des caractéristiques

Les arbres de décision sont faciles à interpréter et peuvent gérer à la fois des données numériques et catégorielles sans avoir besoin de beaucoup de prétraitement. Le pruning est essentiel pour éviter le sur-apprentissage et améliorer la généralisation du modèle.